# By-beat split against patient-wise split, on identical data (5 seeds)

**Question.** How much of the usual ~98% on MIT-BIH comes from having the same patients in train and test?

**Design.** Everything is held fixed except the split:

- the same beats (cut from the raw PhysioNet records, 4 AAMI classes N, S, V, F, paced recordings excluded)
- the same three models, augmentation, class weights and training settings
- the same train / validation / test sizes

| | Train | Validation | Test |
|---|---|---|---|
| **Patient-wise** | 17 DS1 recordings | 5 other DS1 recordings | 22 DS2 recordings |
| **By-beat** | random beats from all 44 recordings | random beats | random beats |

Each setting is run with 5 seeds, so the result is a mean and a standard deviation.

**Difference from the first patient-wise run:** validation there used beats of the training patients. Here it uses
5 held-out recordings, which is the correct protocol.

**Before running on Kaggle:** GPU on, Internet on. About 3 to 4 hours.

In [ ]:
!pip -q install wfdb

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import wfdb
from scipy.signal import resample_poly
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, MaxPooling1D,
    Dense, Dropout, SpatialDropout1D, Flatten, Concatenate,
    GlobalAveragePooling1D, Add, MultiHeadAttention, LayerNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

SEEDS = [int(s) for s in os.environ.get('SEEDS', '0,1,2,3,4').split(',')]

CLASS_NAMES = ['Normal', 'Supraventricular', 'Ventricular', 'Fusion']
NUM_CLASSES = 4
BEAT_LEN    = 187    # samples per beat at 125 Hz, same as the Kaggle files
TARGET_FS   = 125

# quick settings (leave as they are for the real run)
EPOCHS      = int(os.environ.get('EPOCHS', 50))
DATA_DIR    = os.environ.get('MITDB_DIR', 'mitdb')

print(f'TensorFlow {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 1. Beat extraction (same as the first patient-wise notebook)

In [ ]:
# DS1 / DS2 record lists from de Chazal et al. (2004). 22 recordings each.
# The paced recordings 102, 104, 107 and 217 are in neither list.
DS1 = [101, 106, 108, 109, 112, 114, 115, 116, 118, 119, 122, 124,
       201, 203, 205, 207, 208, 209, 215, 220, 223, 230]
DS2 = [100, 103, 105, 111, 113, 117, 121, 123, 200, 202, 210, 212,
       213, 214, 219, 221, 222, 228, 231, 232, 233, 234]
assert len(DS1) == 22 and len(DS2) == 22 and not set(DS1) & set(DS2)

# MIT-BIH beat symbols -> AAMI class index (N=0, S=1, V=2, F=3). Q beats are skipped.
AAMI = {}
for sym in ['N', 'L', 'R', 'e', 'j']:
    AAMI[sym] = 0
for sym in ['A', 'a', 'J', 'S']:
    AAMI[sym] = 1
for sym in ['V', 'E']:
    AAMI[sym] = 2
AAMI['F'] = 3
Q_SYMBOLS = ['/', 'f', 'Q']


def extract_beats(record_path):
    """Cut one record into beats. Returns (X, y, n_skipped_q).

    X has shape (n_beats, 187): each beat starts at its annotated R-peak, is
    1.2 x the median R-R interval long, is scaled to 0..1 and zero-padded.
    """
    record = wfdb.rdrecord(record_path, channel_names=['MLII'])
    ann = wfdb.rdann(record_path, 'atr')
    fs = int(record.fs)

    signal = resample_poly(record.p_signal[:, 0], TARGET_FS, fs)       # 360 Hz -> 125 Hz
    peaks = np.round(ann.sample * TARGET_FS / fs).astype(int)
    symbols = np.array(ann.symbol)

    is_beat = np.isin(symbols, list(AAMI) + Q_SYMBOLS)
    beat_peaks, beat_symbols = peaks[is_beat], symbols[is_beat]
    median_rr = np.median(np.diff(beat_peaks))
    window = int(min(BEAT_LEN, round(1.2 * median_rr)))

    X, y, skipped_q = [], [], 0
    for peak, sym in zip(beat_peaks, beat_symbols):
        if sym in Q_SYMBOLS:
            skipped_q += 1
            continue
        segment = signal[peak:peak + window]
        if len(segment) < window:
            continue                                   # beat cut off by the end of the record
        low, high = segment.min(), segment.max()
        if high - low < 1e-6:
            continue                                   # flat line, nothing to learn
        beat = np.zeros(BEAT_LEN, dtype=np.float32)
        beat[:window] = (segment - low) / (high - low)
        X.append(beat)
        y.append(AAMI[sym])
    return np.array(X, dtype=np.float32).reshape(-1, BEAT_LEN), np.array(y, dtype=int), skipped_q


def load_split(records, data_dir):
    X_all, y_all, groups, skipped = [], [], [], 0
    for rec in records:
        X, y, q = extract_beats(os.path.join(data_dir, str(rec)))
        X_all.append(X); y_all.append(y); groups.append(np.full(len(y), rec)); skipped += q
    return np.vstack(X_all), np.concatenate(y_all), np.concatenate(groups), skipped

In [ ]:
if not os.path.exists(os.path.join(DATA_DIR, '100.hea')):
    print('Downloading the MIT-BIH Arrhythmia Database from PhysioNet...')
    wfdb.dl_database('mitdb', dl_dir=DATA_DIR)

# Validation recordings for the patient-wise setting, taken out of DS1.
# Chosen so that validation contains all four classes while record 208 and 209,
# which hold most of the Fusion and Supraventricular beats, stay in training.
VAL_RECORDS = [106, 116, 124, 215, 223]
TRAIN_RECORDS = [r for r in DS1 if r not in VAL_RECORDS]

X_tr, y_tr, g_tr, _ = load_split(TRAIN_RECORDS, DATA_DIR)
X_va, y_va, g_va, _ = load_split(VAL_RECORDS, DATA_DIR)
X_te, y_te, g_te, _ = load_split(DS2, DATA_DIR)
assert not (set(g_tr) & set(g_va)) and not (set(g_tr) & set(g_te)) and not (set(g_va) & set(g_te))

print(f"{'Class':20s} {'train':>8s} {'val':>8s} {'test':>8s}")
for i, name in enumerate(CLASS_NAMES):
    print(f'{name:20s} {np.sum(y_tr == i):8d} {np.sum(y_va == i):8d} {np.sum(y_te == i):8d}')
print(f"{'Total':20s} {len(y_tr):8d} {len(y_va):8d} {len(y_te):8d}")

PATIENT_WISE = (X_tr, y_tr, X_va, y_va, X_te, y_te)


def by_beat_split(seed):
    """Random stratified split of ALL beats, with the same three sizes as the patient-wise split."""
    X_all = np.vstack([X_tr, X_va, X_te])
    y_all = np.concatenate([y_tr, y_va, y_te])
    X_rest, X_test, y_rest, y_test = train_test_split(
        X_all, y_all, test_size=len(y_te), stratify=y_all, random_state=seed)
    X_train, X_val, y_train, y_val = train_test_split(
        X_rest, y_rest, test_size=len(y_va), stratify=y_rest, random_state=seed)
    return X_train, y_train, X_val, y_val, X_test, y_test

## 2. Augmentation and models (copied unchanged)

In [ ]:
def augment_beat(signal, shift_range=10, scale_range=0.12, noise_std=0.01):
    """
    Apply three ECG-valid augmentations to a single 187-timestep beat.
    
    Args:
        signal      : 1D array of length 187
        shift_range : max time-shift in timesteps (±10 = ±80ms at 125Hz)
        scale_range : amplitude scaling ± this fraction
        noise_std   : standard deviation of additive Gaussian noise
    Returns:
        Augmented signal of same length 187
    """
    # 1. Random time-shift — roll the signal and zero-pad the wrapped edge
    shift = np.random.randint(-shift_range, shift_range)
    aug   = np.roll(signal, shift)
    if shift > 0:
        aug[:shift] = 0.0
    elif shift < 0:
        aug[shift:] = 0.0

    # 2. Random amplitude scaling (multiplicative, mean=1.0)
    scale = 1.0 + np.random.uniform(-scale_range, scale_range)
    aug   = aug * scale

    # 3. Additive Gaussian noise — simulates electrode noise
    aug = aug + np.random.normal(0, noise_std, aug.shape)

    # Clip to valid range [0, 1] (MIT-BIH signals are pre-normalised)
    return np.clip(aug, 0.0, 1.0)


def augment_minority_classes(X, y, target_count=5000, aug_classes=(1, 2, 3)):
    """
    Over-sample minority classes up to target_count by augmentation.
    
    Args:
        X             : raw signals, shape (N, 187)
        y             : integer labels, shape (N,)
        target_count  : desired minimum samples per minority class
        aug_classes   : which class indices to augment (S=1, V=2, F=3)
    Returns:
        X_aug, y_aug  : augmented dataset (original + synthetic samples)
    """
    X_extra, y_extra = [], []
    for cls in aug_classes:
        mask      = (y == cls)
        X_cls     = X[mask]           # all real samples for this class
        n_real    = len(X_cls)
        n_needed  = max(0, target_count - n_real)
        if n_needed == 0:
            continue
        # Randomly pick which real samples to augment (with replacement)
        idxs = np.random.choice(n_real, n_needed, replace=True)
        for idx in idxs:
            X_extra.append(augment_beat(X_cls[idx]))
            y_extra.append(cls)

    if X_extra:
        X_aug = np.vstack([X, np.array(X_extra, dtype=np.float32)])
        y_aug = np.concatenate([y, np.array(y_extra, dtype=int)])
    else:
        X_aug, y_aug = X, y
    return X_aug, y_aug

In [ ]:
# ──────────────────────────────────────────────────────────────
# SHARED BUILDING BLOCKS & EVALUATION UTILITIES
# ──────────────────────────────────────────────────────────────

def dense_block(x, filters):
    """
    DenseNet block: each layer receives all previous feature maps.
    Output = concatenation of [input, conv1_out, conv2_out].
    This dense connectivity provides two benefits:
      - Feature reuse: early low-level ECG features survive to later layers
      - Implicit deep supervision: gradients flow directly to early layers
    """
    c1 = Conv1D(filters, 3, padding='same', activation='relu')(x)
    c1 = BatchNormalization()(c1)
    c1 = SpatialDropout1D(0.1)(c1)   # Drops entire feature maps, more effective than Dropout
    m1 = Concatenate()([x, c1])

    c2 = Conv1D(filters, 3, padding='same', activation='relu')(m1)
    c2 = BatchNormalization()(c2)
    c2 = SpatialDropout1D(0.1)(c2)
    return Concatenate()([x, c1, c2])


def transition_layer(x, out_filters):
    """
    Transition layer between dense blocks.
    Problem it solves: dense_block output has x + c1 + c2 channels.
    After 3 blocks (32→64→128), you'd have ~500 channels WITHOUT this.
    The 1×1 conv compresses back to out_filters (halving), keeping
    the model lean and reducing overfitting risk.
    This is the standard DenseNet-BC design (Huang et al., 2017).
    """
    x = Conv1D(out_filters, 1, padding='same', activation='relu')(x)  # 1×1 compression
    x = BatchNormalization()(x)
    return MaxPooling1D(2)(x)  # Halve temporal resolution


def attention_block(x, num_heads=4, key_dim=16, name_prefix='attn'):
    """
    Multi-Head Self-Attention block with residual connection.
    
    How it helps ECG classification:
    - Attention heads learn to focus on different ECG regions simultaneously
      (one head for QRS complex, another for T-wave, etc.)
    - The residual Add() preserves original features so the block
      is additive — it can only enhance, never destroy, existing features
    - LayerNorm stabilises training by normalising across feature dimension
    """
    # Project to attention dimension first (reduces computation)
    proj = Conv1D(num_heads * key_dim, 1, padding='same', activation='relu',
                  name=f'{name_prefix}_proj')(x)

    # Self-attention: Q=K=V=proj (the signal attends to itself)
    attn = MultiHeadAttention(
        num_heads=num_heads, key_dim=key_dim,
        dropout=0.1, name=f'{name_prefix}_mha'
    )(proj, proj)

    # Project back to original channel size for the residual addition
    attn = Conv1D(x.shape[-1], 1, padding='same', name=f'{name_prefix}_back')(attn)

    # Residual connection + LayerNormalization
    x = Add(name=f'{name_prefix}_add')([x, attn])
    return LayerNormalization(name=f'{name_prefix}_ln')(x)

In [ ]:
def build_cnn():
    inp = Input(shape=(187, 1), name='ecg_input')

    # Block 1: broad filters (size 5) to capture QRS and early repolarisation
    x = Conv1D(32, 5, activation='relu', padding='same')(inp)
    x = BatchNormalization()(x)
    x = SpatialDropout1D(0.1)(x)   # Drop 10% of feature maps per step
    x = MaxPooling1D(2)(x)          # 187 → 93 timesteps

    # Block 2: medium filters to capture QRS complex shape
    x = Conv1D(64, 5, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = SpatialDropout1D(0.1)(x)
    x = MaxPooling1D(2)(x)          # 93 → 46 timesteps

    # Block 3: narrow filters (size 3) for fine T-wave morphology
    x = Conv1D(128, 3, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = SpatialDropout1D(0.1)(x)
    x = MaxPooling1D(2)(x)          # 46 → 23 timesteps

    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)             # Standard dropout in dense head
    out = Dense(NUM_CLASSES, activation='softmax')(x)

    return Model(inp, out, name='CNN_Baseline')

In [ ]:
def build_densenet_improved():
    inp = Input(shape=(187, 1), name='ecg_input')

    # ── Stage 1 ────────────────────────────────────────────────
    # Initial conv to establish 32-channel feature space
    x = Conv1D(32, 5, padding='same', activation='relu')(inp)
    x = BatchNormalization()(x)
    # Dense block: each conv sees ALL previous outputs
    x = dense_block(x, 32)
    # Transition: compress back to 32 channels and halve time resolution
    # Without this, the concatenated output would be ~96 channels here
    x = transition_layer(x, out_filters=32)

    # ── Stage 2 ────────────────────────────────────────────────
    x = Conv1D(64, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = dense_block(x, 64)
    x = transition_layer(x, out_filters=64)  # Compresses ~192 → 64

    # ── Stage 3 ────────────────────────────────────────────────
    x = Conv1D(128, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = dense_block(x, 128)
    x = transition_layer(x, out_filters=128)  # Compresses ~384 → 128

    # Global average pooling collapses the temporal dimension
    # to a single 128-dimensional feature vector per beat
    x = GlobalAveragePooling1D()(x)

    # Dense classification head with L2 regularisation + dropout
    x = Dense(128, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = Dropout(0.4)(x)
    x = Dense(64, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = Dropout(0.3)(x)
    out = Dense(NUM_CLASSES, activation='softmax')(x)

    return Model(inp, out, name='DenseNet_Improved')

In [ ]:
def build_proposed_model():
    inp = Input(shape=(187, 1), name='ecg_input')

    # ── Stage 1: Low-level feature extraction ─────────────────
    # Broad kernel (5) captures slow waveform components (ST segment and T-wave)
    x = Conv1D(32, 5, padding='same', activation='relu')(inp)
    x = BatchNormalization()(x)
    x = dense_block(x, 32)
    x = transition_layer(x, out_filters=32)   # 187 → 93 timesteps

    # ── Stage 2: Mid-level feature extraction ─────────────────
    x = Conv1D(64, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = dense_block(x, 64)
    x = transition_layer(x, out_filters=64)   # 93 → 46 timesteps

    # ── ATTENTION BLOCK 1: Mid-scale (46 timesteps ≈ 128ms) ───
    # At this resolution, the model can learn which temporal segments
    # (PR interval, QRS onset, ST segment) are most discriminative
    # for each arrhythmia class.
    # 4 heads × key_dim 16 = 64 attention dimensions
    x = attention_block(x, num_heads=4, key_dim=16, name_prefix='attn1')

    # ── Stage 3: High-level feature extraction ────────────────
    x = Conv1D(128, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = dense_block(x, 128)
    x = transition_layer(x, out_filters=128)  # 46 → 23 timesteps

    # ── ATTENTION BLOCK 2: Fine-scale (23 timesteps ≈ 64ms) ───
    # At this compressed resolution, attention learns which of the
    # high-level features (e.g., QRS notch, ST elevation) are
    # diagnostically critical for the final classification.
    # 8 heads for richer attention diversity at the decision layer.
    x = attention_block(x, num_heads=8, key_dim=16, name_prefix='attn2')

    # Global average pooling → 128-dim beat representation
    x = GlobalAveragePooling1D()(x)

    # ── Classification head ───────────────────────────────────
    x = Dense(128, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = Dropout(0.4)(x)
    x = Dense(64, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = Dropout(0.3)(x)
    out = Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

    return Model(inp, out, name='DenseNet_MultiScale_Attention')

## 3. Training loop

Settings as in the main notebook: Adam, categorical cross-entropy, batch size 64, up to 50 epochs,
early stopping on validation accuracy (patience 7), learning rate halved on a plateau.
Only the training part is augmented.

In [ ]:
MODELS = [
    ('CNN baseline', build_cnn, 0.001, None),
    ('DenseNet', build_densenet_improved, 0.0005, 1.0),
    ('DenseNet + attention', build_proposed_model, 0.0005, 1.0),
]
LABELS = list(range(NUM_CLASSES))
rows, confusion = [], {}


def run_one(split_name, data, model_spec, seed):
    name, build_fn, learning_rate, clipnorm = model_spec
    X_train, y_train, X_val, y_val, X_test, y_test = data

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)          # seeds Python, NumPy and TensorFlow

    X_aug, y_aug = augment_minority_classes(X_train, y_train, target_count=5000)
    weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_aug)
    X_fit, y_fit = shuffle(X_aug.reshape(-1, BEAT_LEN, 1),
                           tf.keras.utils.to_categorical(y_aug, NUM_CLASSES), random_state=seed)

    model = build_fn()
    optimizer = Adam(learning_rate=learning_rate, clipnorm=clipnorm) if clipnorm else Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(
        X_fit, y_fit, epochs=EPOCHS, batch_size=64, verbose=0,
        validation_data=(X_val.reshape(-1, BEAT_LEN, 1), tf.keras.utils.to_categorical(y_val, NUM_CLASSES)),
        class_weight=dict(enumerate(weights)),
        callbacks=[
            EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, min_lr=1e-7),
        ],
    )
    y_pred = np.argmax(model.predict(X_test.reshape(-1, BEAT_LEN, 1), verbose=0), axis=1)
    report = classification_report(y_test, y_pred, labels=LABELS, target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    row = {'split': split_name, 'model': name, 'seed': seed,
           'epochs': len(history.history['loss']),
           'best_val_accuracy': max(history.history['val_accuracy']),
           'accuracy': report['accuracy'], 'macro_f1': report['macro avg']['f1-score']}
    for cls in CLASS_NAMES:
        row[f'{cls} F1'] = report[cls]['f1-score']
        row[f'{cls} recall'] = report[cls]['recall']
        row[f'{cls} precision'] = report[cls]['precision']
    rows.append(row)
    confusion[(split_name, name, seed)] = confusion_matrix(y_test, y_pred, labels=LABELS)
    print(f"{split_name:13s} {name:22s} seed {seed}: accuracy {row['accuracy']:.4f}  macro F1 {row['macro_f1']:.4f}  "
          f"best val acc {row['best_val_accuracy']:.4f}  ({row['epochs']} epochs)", flush=True)
    pd.DataFrame(rows).to_csv('all_runs.csv', index=False)       # saved after every run


for seed in SEEDS:
    for spec in MODELS:
        run_one('patient-wise', PATIENT_WISE, spec, seed)
        run_one('by-beat', by_beat_split(seed), spec, seed)

## 4. Results

In [ ]:
runs = pd.DataFrame(rows)
order = [m[0] for m in MODELS]


def mean_std(frame, column):
    g = frame.groupby(['model', 'split'])[column].agg(['mean', 'std']).round(4)
    return (g['mean'].map('{:.3f}'.format) + ' ± ' + g['std'].fillna(0).map('{:.3f}'.format)).unstack('split').loc[order]


print(f'Seeds: {SEEDS}\n')
print('Macro F1 (mean ± std over seeds)');  display(mean_std(runs, 'macro_f1')[['by-beat', 'patient-wise']])
print('Accuracy (mean ± std over seeds)');  display(mean_std(runs, 'accuracy')[['by-beat', 'patient-wise']])
print('Best validation accuracy');          display(mean_std(runs, 'best_val_accuracy')[['by-beat', 'patient-wise']])

for cls in CLASS_NAMES:
    print(f'{cls}: F1 (mean ± std)')
    display(mean_std(runs, f'{cls} F1')[['by-beat', 'patient-wise']])

summary = runs.groupby(['split', 'model']).mean(numeric_only=True).drop(columns=['seed']).round(4)
summary.to_csv('summary_mean.csv')
runs.groupby(['split', 'model']).std(numeric_only=True).drop(columns=['seed']).round(4).to_csv('summary_std.csv')

In [ ]:
# Confusion matrices of the first seed, patient-wise against by-beat, for each model
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for col, name in enumerate(order):
    for row_i, split_name in enumerate(['by-beat', 'patient-wise']):
        cm = confusion[(split_name, name, SEEDS[0])]
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[row_i, col],
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        axes[row_i, col].set_title(f'{name}, {split_name} (seed {SEEDS[0]})')
        axes[row_i, col].set_xlabel('Predicted'); axes[row_i, col].set_ylabel('True')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120)
plt.show()